# Train OSNet on MEVID with KaggleHub

Enable **GPU** and **Internet**, set `KAGGLE_DATASET_HANDLE` in Cell 1, then run all cells from top to bottom.
KaggleHub makes MEVID available before any dataset discovery, FastReID setup, OSNet configuration, training,
or evaluation runs.

## Cell 1 — Basic imports and configuration

In [ ]:
from pathlib import Path
import os
import sys
import json
import shutil
import subprocess
import importlib.util

if importlib.util.find_spec("kagglehub") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "kagglehub"], check=True)
import kagglehub

# Replace this with the MEVID Kaggle Dataset handle, for example: owner/dataset-name
KAGGLE_DATASET_HANDLE = "OWNER/DATASET-SLUG"

WORK = Path("/kaggle/working/osnet_mevid")
OUT = WORK / "results/osnet"
EPOCHS = 60
BATCH_SIZE = 64
EVAL_EVERY = 10
NUM_WORKERS = 2
FRAME_STEP = 20
SEED = 42
LOG_EVERY = 10

PRETRAINED_WEIGHTS = ""
RESUME_CHECKPOINT = ""
AUTO_RESUME = True

assert EPOCHS >= 1
assert BATCH_SIZE >= 8 and BATCH_SIZE % 4 == 0
assert FRAME_STEP >= 1 and EVAL_EVERY >= 1 and NUM_WORKERS >= 0
WORK.mkdir(parents=True, exist_ok=True)
OUT.mkdir(parents=True, exist_ok=True)
print("Output directory:", OUT)

## Cell 2 — Attach/download MEVID with KaggleHub

`dataset_download()` uses KaggleHub's managed cache and returns the dataset path. The notebook trains directly
from that returned path and never writes to `/kaggle/input`.

In [ ]:
handle = KAGGLE_DATASET_HANDLE.strip()
if (
    not handle
    or handle.upper() == "OWNER/DATASET-SLUG"
    or "/" not in handle
):
    raise RuntimeError(
        "Set KAGGLE_DATASET_HANDLE to the MEVID Kaggle Dataset handle, "
        "for example owner/dataset-name."
    )

try:
    MEVID_ROOT = Path(kagglehub.dataset_download(handle)).resolve()
except Exception as exc:
    raise RuntimeError(
        "Failed to load MEVID through KaggleHub. "
        "Check KAGGLE_DATASET_HANDLE and dataset permissions."
    ) from exc

if not MEVID_ROOT.is_dir():
    raise RuntimeError(f"KaggleHub returned a missing or invalid directory: {MEVID_ROOT}")
print("KaggleHub MEVID path:", MEVID_ROOT)

## Cell 3 — Detect and validate the MEVID structure

In [ ]:
REQUIRED_FOLDER_NAMES = {
    "bbox_train",
    "bbox_test",
    "mevid-v1-annotation-data",
}


def find_unique_directory(root: Path, folder_name: str) -> Path:
    matches = []
    for current, directories, _ in os.walk(root):
        current_path = Path(current)
        for directory in list(directories):
            if directory == folder_name:
                matches.append((current_path / directory).resolve())
            # Do not walk through the very large MEVID image directories.
            if directory in REQUIRED_FOLDER_NAMES:
                directories.remove(directory)

    matches = list(dict.fromkeys(matches))
    if len(matches) == 0:
        raise RuntimeError(f"Could not find {folder_name!r} under {root}")
    if len(matches) > 1:
        raise RuntimeError(f"Found multiple {folder_name!r} directories: {matches}")
    return matches[0]


bbox_train_dir = find_unique_directory(MEVID_ROOT, "bbox_train")
bbox_test_dir = find_unique_directory(MEVID_ROOT, "bbox_test")
annotation_dir = find_unique_directory(MEVID_ROOT, "mevid-v1-annotation-data")

DATA_DIRS = {
    "bbox_train": str(bbox_train_dir),
    "bbox_test": str(bbox_test_dir),
    "annotation": str(annotation_dir),
}

for name, path in DATA_DIRS.items():
    path = Path(path)
    if not path.is_dir():
        raise RuntimeError(f"MEVID directory does not exist: {name}: {path}")

required_annotations = (
    "train_name.txt",
    "test_name.txt",
    "track_train_info.txt",
    "track_test_info.txt",
    "query_IDX.txt",
)
for filename in required_annotations:
    annotation_file = annotation_dir / filename
    if not annotation_file.is_file():
        raise RuntimeError(f"Missing MEVID annotation file: {annotation_file}")

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp"}


def count_images(directory):
    count = 0
    for _, _, filenames in os.walk(directory):
        count += sum(
            Path(filename).suffix.lower() in IMAGE_EXTENSIONS
            for filename in filenames
        )
    return count


TRAIN_IMAGE_COUNT = count_images(DATA_DIRS["bbox_train"])
TEST_IMAGE_COUNT = count_images(DATA_DIRS["bbox_test"])
if TRAIN_IMAGE_COUNT == 0 or TEST_IMAGE_COUNT == 0:
    raise RuntimeError(
        f"MEVID image count is invalid: train={TRAIN_IMAGE_COUNT}, test={TEST_IMAGE_COUNT}"
    )

print("====================================")
print("MEVID DATASET READY")
print("====================================")
for name, path in DATA_DIRS.items():
    print(f"{name}: {path}")
print("Training images:", TRAIN_IMAGE_COUNT)
print("Testing images:", TEST_IMAGE_COUNT)
print("Dataset validation passed.")

## Cell 4 — FastReID and OSNet environment setup

In [ ]:
print("Dataset validation passed.")
print("Starting FastReID / OSNet setup...")

import torch
import torchvision

if not torch.cuda.is_available():
    raise RuntimeError("Enable a GPU in Kaggle Settings > Accelerator, restart, then Run All.")

print("PyTorch:", torch.__version__)
print("torchvision:", torchvision.__version__)
print("GPU:", torch.cuda.get_device_name(0))

packages = {
    "yacs": "yacs",
    "termcolor": "termcolor",
    "prettytable": "prettytable",
    "easydict": "easydict",
    "gdown": "gdown",
    "faiss": "faiss-cpu",
    "tensorboard": "tensorboard",
    "tabulate": "tabulate",
    "tqdm": "tqdm",
    "sklearn": "scikit-learn",
    "yaml": "PyYAML",
    "scipy": "scipy",
}
missing = [package for module, package in packages.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)

FASTREID_COMMIT = "c9bc3ceb2f7a6438b62fb515ea3df6d1e999e95d"
FASTREID_ROOT = WORK / "fast-reid/upstream"
if not (FASTREID_ROOT / ".git").exists():
    FASTREID_ROOT.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(
        ["git", "clone", "--quiet", "https://github.com/JDAI-CV/fast-reid.git", str(FASTREID_ROOT)],
        check=True,
    )
subprocess.run(
    ["git", "-C", str(FASTREID_ROOT), "checkout", "--quiet", "--detach", FASTREID_COMMIT],
    check=True,
)
sys.path.insert(0, str(FASTREID_ROOT))
os.environ["FASTREID_DATASETS"] = str(MEVID_ROOT)
os.environ["TORCH_HOME"] = str(WORK / "weights/torch")

if PRETRAINED_WEIGHTS:
    source = Path(PRETRAINED_WEIGHTS)
    if not source.is_file():
        raise FileNotFoundError(source)
    destination = WORK / "weights/torch/checkpoints/osnet_x1_0_imagenet.pth"
    destination.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(source, destination)
print("FastReID ready:", FASTREID_ROOT)

## Cell 5 — Build MEVID train/query/gallery splits

In [ ]:
from tqdm.auto import tqdm


def load_mevid_splits(data_dirs, frame_step=20):
    annotations = Path(data_dirs["annotation"])
    image_roots = {
        "train": Path(data_dirs["bbox_train"]),
        "test": Path(data_dirs["bbox_test"]),
    }
    query_rows = {int(float(value)) for value in (annotations / "query_IDX.txt").read_text().split()}
    splits = {"train": [], "query": [], "gallery": []}

    for subset in ("train", "test"):
        names = (annotations / f"{subset}_name.txt").read_text().splitlines()
        tracks = (annotations / f"track_{subset}_info.txt").read_text().splitlines()
        for row, line in enumerate(tqdm(tracks, desc=f"Reading {subset}", unit="track")):
            start, end, pid, outfit, camera = [int(float(value)) for value in line.split()]
            if end == start - 1:
                continue
            if not 0 <= start <= end < len(names):
                raise ValueError(f"Invalid {subset} tracklet {row}: {start}, {end}")
            split = "train" if subset == "train" else ("query" if row in query_rows else "gallery")
            for index in range(start, end + 1, frame_step):
                image = image_roots[subset] / f"{pid:04d}" / names[index].strip()
                if not image.is_file():
                    raise FileNotFoundError(image)
                splits[split].append((str(image), pid, camera))

    for name, samples in splits.items():
        if not samples:
            raise RuntimeError(f"MEVID {name} split is empty")
        identities = len({sample[1] for sample in samples})
        print(f"{name:7s}: {len(samples):,} images, {identities} identities")
    return splits


SPLITS = load_mevid_splits(DATA_DIRS, FRAME_STEP)

## Cell 6 — Define the FastReID dataset, progress display, and evaluator

In [ ]:
import collections
import collections.abc
collections.Mapping = collections.abc.Mapping
collections.Iterable = collections.abc.Iterable

from fastreid.data.datasets import DATASET_REGISTRY
from fastreid.data.datasets.bases import ImageDataset
from fastreid.engine import DefaultTrainer, hooks
from fastreid.engine.train_loop import HookBase
from fastreid.evaluation import ReidEvaluator


class MEVID_OSNet(ImageDataset):
    def __init__(self, root=None, **kwargs):
        super().__init__(SPLITS["train"], SPLITS["query"], SPLITS["gallery"], **kwargs)


if "MEVID_OSNet" not in DATASET_REGISTRY._obj_map:
    DATASET_REGISTRY.register(MEVID_OSNet)


class NotebookProgress(HookBase):
    bar = None

    def before_epoch(self):
        self.bar = tqdm(
            total=self.trainer.iters_per_epoch,
            desc=f"Epoch {self.trainer.epoch + 1}/{self.trainer.max_epoch}",
            unit="batch",
        )

    def after_step(self):
        latest = self.trainer.storage.latest()
        loss = latest.get("total_loss")
        if loss is not None:
            self.bar.set_postfix(loss=f"{loss[0]:.4f}", refresh=False)
        self.bar.update(1)

    def after_epoch(self):
        if self.bar is not None:
            self.bar.close()

    def after_train(self):
        if self.bar is not None:
            self.bar.close()


class OSNetEvaluator(ReidEvaluator):
    def _compile_dependencies(self):
        pass  # use FastReID's NumPy evaluator; no runtime Cython build


class OSNetTrainer(DefaultTrainer):
    def build_writers(self):
        from fastreid.utils.events import JSONWriter, TensorboardXWriter
        return [
            JSONWriter(os.path.join(self.cfg.OUTPUT_DIR, "metrics.json")),
            TensorboardXWriter(self.cfg.OUTPUT_DIR),
        ]

    def build_hooks(self):
        trainer_hooks = super().build_hooks()
        for hook in trainer_hooks:
            if isinstance(hook, hooks.PeriodicWriter):
                hook._period = LOG_EVERY
        return [NotebookProgress(), *trainer_hooks]

    @classmethod
    def build_evaluator(cls, cfg, dataset_name, output_dir=None):
        loader, num_query = cls.build_test_loader(cfg, dataset_name)
        return loader, OSNetEvaluator(cfg, num_query, output_dir)

## Cell 7 — Configure OSNet

In [ ]:
from types import SimpleNamespace
from fastreid.config import get_cfg
from fastreid.engine import default_setup
import fastreid.data.build as data_build


# Use a standard DataLoader so worker exceptions are shown directly in Kaggle.
def standard_loader(local_rank, **kwargs):
    kwargs["pin_memory"] = True
    kwargs["num_workers"] = NUM_WORKERS
    return torch.utils.data.DataLoader(**kwargs)


data_build.DataLoaderX = standard_loader

cfg = get_cfg()
cfg.DATASETS.NAMES = ("MEVID_OSNet",)
cfg.DATASETS.TESTS = ("MEVID_OSNet",)
cfg.MODEL.DEVICE = "cuda"
cfg.MODEL.BACKBONE.NAME = "build_osnet_backbone"
cfg.MODEL.BACKBONE.DEPTH = "x1_0"
cfg.MODEL.BACKBONE.FEAT_DIM = 512
cfg.MODEL.BACKBONE.PRETRAIN = True
cfg.MODEL.BACKBONE.PRETRAIN_PATH = ""
cfg.MODEL.BACKBONE.WITH_IBN = False
cfg.MODEL.HEADS.NAME = "EmbeddingHead"
cfg.MODEL.HEADS.NORM = "BN"
cfg.MODEL.HEADS.POOL_LAYER = "GeneralizedMeanPoolingP"
cfg.MODEL.HEADS.EMBEDDING_DIM = 512
cfg.MODEL.LOSSES.NAME = ("CrossEntropyLoss", "TripletLoss")
cfg.MODEL.LOSSES.CE.EPSILON = 0.1
cfg.MODEL.LOSSES.TRI.MARGIN = 0.3
cfg.MODEL.LOSSES.TRI.HARD_MINING = True

cfg.INPUT.SIZE_TRAIN = [256, 128]
cfg.INPUT.SIZE_TEST = [256, 128]
cfg.INPUT.REA.ENABLED = True
cfg.INPUT.REA.PROB = 0.5
cfg.INPUT.FLIP.ENABLED = True
cfg.INPUT.FLIP.PROB = 0.5
cfg.INPUT.PADDING.ENABLED = True
cfg.INPUT.PADDING.SIZE = 10
cfg.INPUT.CJ.ENABLED = True
cfg.INPUT.CJ.PROB = 0.5

cfg.DATALOADER.NUM_INSTANCE = 4
cfg.DATALOADER.NUM_WORKERS = NUM_WORKERS
cfg.DATALOADER.SAMPLER_TRAIN = "BalancedIdentitySampler"
cfg.SOLVER.OPT = "Adam"
cfg.SOLVER.BASE_LR = 0.00035
cfg.SOLVER.WEIGHT_DECAY = 0.0005
cfg.SOLVER.WEIGHT_DECAY_BIAS = 0.0005
cfg.SOLVER.IMS_PER_BATCH = BATCH_SIZE
cfg.SOLVER.MAX_EPOCH = EPOCHS
cfg.SOLVER.WARMUP_ITERS = 2000
cfg.SOLVER.WARMUP_METHOD = "linear"
cfg.SOLVER.STEPS = [40, 55]
cfg.SOLVER.CHECKPOINT_PERIOD = 1

cfg.TEST.EVAL_PERIOD = EVAL_EVERY
cfg.TEST.IMS_PER_BATCH = 128
cfg.TEST.METRIC = "cosine"
cfg.TEST.RERANK.ENABLED = False
cfg.OUTPUT_DIR = str(OUT)
cfg.SEED = SEED
cfg.freeze()

setup_args = SimpleNamespace(config_file="", eval_only=False, resume=AUTO_RESUME)
default_setup(cfg, setup_args)
print(f"OSNet-x1.0 | {EPOCHS} epochs | batch {BATCH_SIZE} | {torch.cuda.get_device_name(0)}")

## Cell 8 — Train and evaluate

In [ ]:
last_checkpoint = OUT / "last_checkpoint"
if RESUME_CHECKPOINT and not last_checkpoint.exists():
    source = Path(RESUME_CHECKPOINT)
    if not source.is_file():
        raise FileNotFoundError(source)
    destination = OUT / source.name
    shutil.copy2(source, destination)
    last_checkpoint.write_text(destination.name, encoding="utf-8")

resume = AUTO_RESUME and last_checkpoint.exists()
if last_checkpoint.exists() and not resume:
    raise RuntimeError("A checkpoint exists. Enable AUTO_RESUME or select a new WORK directory.")

trainer = OSNetTrainer(cfg)
trainer.resume_or_load(resume=resume)
print("Resuming latest completed epoch." if resume else "Starting ImageNet-pretrained OSNet training.")
FINAL_METRICS = trainer.train()

result_path = OUT / "evaluation.json"
result_path.write_text(
    json.dumps(FINAL_METRICS, indent=2, default=lambda value: value.item()),
    encoding="utf-8",
)
print("Final evaluation:", result_path)

## Cell 9 — Show and download results

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import zipfile
from IPython.display import FileLink, display

result_path = OUT / "evaluation.json"
if not result_path.is_file():
    raise RuntimeError("Finish the training cell before opening results.")

metrics = json.loads(result_path.read_text(encoding="utf-8"))
display(pd.DataFrame([metrics], index=["OSNet-x1.0 / MEVID"]))
pd.DataFrame([metrics]).to_csv(OUT / "evaluation.csv", index=False)

history_path = OUT / "metrics.json"
if history_path.is_file():
    records = [json.loads(line) for line in history_path.read_text().splitlines() if line.strip()]
    history = pd.DataFrame(records)
    if {"iteration", "total_loss"}.issubset(history.columns):
        history = history.drop_duplicates("iteration", keep="last").sort_values("iteration")
        ax = history.dropna(subset=["total_loss"]).plot(
            x="iteration", y="total_loss", figsize=(10, 4), grid=True
        )
        ax.set_title("OSNet training loss")
        ax.figure.tight_layout()
        ax.figure.savefig(OUT / "training_loss.png", dpi=150)
        plt.show()

archive = Path("/kaggle/working/osnet_mevid_results.zip")
with zipfile.ZipFile(archive, "w", compression=zipfile.ZIP_DEFLATED) as bundle:
    for path in OUT.rglob("*"):
        if path.is_file() and (path.suffix != ".pth" or path.name in {"model_final.pth", "model_best.pth"}):
            bundle.write(path, arcname=str(Path("osnet") / path.relative_to(OUT)))

os.chdir("/kaggle/working")
display(FileLink(archive.name))
display(FileLink(str(result_path.relative_to(Path.cwd()))))
print("All outputs:", OUT)